# SURGE in one notebook — a fusion surrogate end to end

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asvillar/SURGE/blob/release/0.1.0/examples/notebooks/surge_fusion_quickstart.ipynb)

**SURGE** (Surrogate Unified Robust Generation Engine) trains, tunes, evaluates and exports scientific
surrogate models from one declarative workflow. This notebook replays the four stages of the SURGE
end-to-end video on a real fusion problem, **QLKNN turbulent transport**, and then runs a second fusion
dataset, **SOLPS-NN divertor exhaust**, with an in-context tabular foundation model.

| Stage | What you will run | Video panel |
|---|---|---|
| 1 · Ingest & characterize | `surge init` inspects the dataset and writes a commented spec | dataset characterization |
| 2 · Model & optimize | `surge run` with an Optuna HPO campaign on a residual MLP | live HPO + mission control |
| 3 · Evaluate & quantify | held-out metrics, parity, a multi-model leaderboard | parity, leaderboard vs. gate |
| 4 · Report & deploy | the self-contained HTML report, reload the model and predict | report |

Runtime: about 10 minutes on a Colab CPU with the default 20-trial campaign (the README figure comes
from a 200-trial campaign, 1 h 48 min on a laptop). A GPU runtime makes the PyTorch and Kumo steps faster
but is not required.

In [ ]:
from IPython.display import HTML
HTML('''<video width="820" controls autoplay muted loop
  src="https://raw.githubusercontent.com/asvillar/SURGE/release/0.1.0/docs/assets/readme/surge_end_to_end.mp4">
</video><p><em>The SURGE end-to-end video: the four-stage pipeline on QLKNN transport — this notebook reproduces it stage by stage.</em></p>''')

## 0 · Setup

The cell below clones the repository (for the figure gallery script and example configs) and installs
`surge-ml` with the PyTorch, benchmark and Kumo extras plus `fusion_surrogates`, which generates the QLKNN
dataset. Once the package is on PyPI the first line becomes `pip install "surge-ml[torch,benchmarks,kumo]"`.

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path

# quiet TensorFlow's CPU/CUDA start-up chatter (SURGE only probes it; nothing here uses it)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("GRPC_VERBOSITY", "ERROR")

REF = "release/0.1.0"                     # branch or tag to install from
IN_COLAB = "google.colab" in sys.modules
if not Path("examples/viz_theme_gallery.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", REF, "https://github.com/asvillar/SURGE.git"], check=True)
    os.chdir("SURGE")
if importlib.util.find_spec("surge") is None or IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[torch,benchmarks,kumo]", "fusion_surrogates"], check=True)
# bind the CLI to this kernel's interpreter so a stale `surge` on PATH can never shadow it
SURGE = f"{sys.executable} -m surge.cli"
GALLERY = f"{sys.executable} examples/viz_theme_gallery.py"
print(subprocess.run([sys.executable, "-m", "surge.cli", "version"], capture_output=True, text=True).stdout.strip())
print("working directory:", Path.cwd())

In [ ]:
!{SURGE} models --verbose | tail -n 8

## 1 · Ingest & characterize

`plasma.qlknn_transport` predicts the ITG electron heat flux (`efeITG`, gyro-Bohm units) from ten
normalised gyrokinetic inputs; the ground truth is the published QLKNN_7_11 network queried through
`fusion_surrogates` (van de Plassche et al., *Phys. Plasmas* 2020). A first benchmark call generates and
caches the table; `surge init` then characterizes it and writes a commented workflow spec, the way the
wizard would for your own CSV.

In [ ]:
import numpy as np, pandas as pd
from surge.benchmarks import paths

cache = paths.bench_data_root() / "plasma" / "qlknn_transport.npz"
if not cache.exists():
    subprocess.run([sys.executable, "-m", "surge.cli", "bench", "-b", "plasma.qlknn_transport", "-m", "sklearn.ridge", "--no-save"], check=True)
d = np.load(cache)
cols = ["Ati", "Ate", "Ane", "Ani", "q", "smag", "x", "Ti_Te", "LogNuStar", "normni"]
df = pd.DataFrame(d["X"], columns=cols).assign(efeITG=d["y"])
df.to_csv("qlknn_transport.csv", index=False)
print(df.shape); df.describe().T[["mean", "std", "min", "max"]].round(3)

In [ ]:
!{SURGE} init --data qlknn_transport.csv --target efeITG --goal accuracy --budget standard --yes --out qlknn_spec.yaml
!{SURGE} validate qlknn_spec.yaml

## 2 · Model & optimize — an HPO campaign

The spec below trains a random forest and a residual MLP; the MLP gets an Optuna TPE search over depth,
width, learning rate and dropout, scored on the validation R². `N_TRIALS` is the knob: 20 is a Colab-sized
campaign, 200 is what the README figure shows. Every trial's training history and the search manifest are
written into `runs/<tag>/`, which is what the mission-control dashboard reads.

In [ ]:
N_TRIALS = int(os.environ.get("SURGE_NB_TRIALS", 20))
EPOCHS = int(os.environ.get("SURGE_NB_EPOCHS", 40))
spec = f'''
dataset_path: qlknn_transport.csv
dataset_format: csv
metadata_overrides:
  inputs: [Ati, Ate, Ane, Ani, q, smag, x, Ti_Te, LogNuStar, normni]
  outputs: [efeITG]
test_fraction: 0.2
val_fraction: 0.1
standardize_inputs: true
seed: 42
output_dir: .
run_tag: qlknn_colab
overwrite_existing_run: true
models:
  - key: sklearn.random_forest
    name: qlknn_rf
    params: {{n_estimators: 200, n_jobs: -1}}
  - key: pytorch.residual_mlp
    name: qlknn_residual_mlp
    params: {{n_epochs: {EPOCHS}, patience: 10, verbose: false}}
    hpo:
      enabled: true
      n_trials: {N_TRIALS}
      direction: maximize
      metric: val_r2
      search_space:
        hidden_layers: {{type: categorical, choices: [[128, 128], [256, 128], [256, 256, 128]]}}
        learning_rate: {{type: loguniform, low: 1.0e-4, high: 1.0e-1}}
        dropout_rate: {{type: float, low: 0.0, high: 0.4}}
'''
Path("qlknn_hpo.yaml").write_text(spec)
!{SURGE} validate qlknn_hpo.yaml

In [ ]:
!{SURGE} run qlknn_hpo.yaml 2>&1 | tail -n 25

### Mission control

The dashboard the video assembles trial by trial: per-trial validation curves, the convergence of the
search with the best trial starred, parameter sensitivity, the best trial's train/validation curves and
held-out parity, all rendered from the run's own artifacts by the SURGE visual system.

In [ ]:
!{GALLERY} --only mission_control hpo_convergence training_curves parity \
    --run runs/qlknn_colab --hpo-run runs/qlknn_colab --modes light --out gallery 2>&1 | tail -n 4
from IPython.display import Image, display
for name in ("mission_control", "hpo_convergence", "training_curves", "parity"):
    p = next(Path("gallery").glob(f"{name}*light*.png"), None) or next(Path("gallery").glob(f"{name}*.png"), None)
    if p: display(Image(filename=str(p), width=900))

## 3 · Evaluate & quantify

`runs/qlknn_colab/metrics.json` holds every split metric per model; the predictions are saved as Parquet
and the model cards record what was trained. Next, the benchmark runner scores several registry models on
the same benchmark against its published gate and writes `result.json` receipts that the leaderboard and
the report read; `kumo.tabular_regressor` is NVIDIA's in-context tabular foundation model, which needs no
training at all.

In [ ]:
import json
m = json.load(open("runs/qlknn_colab/metrics.json"))
pd.DataFrame({k: {kk: round(vv, 4) for kk, vv in v.items() if isinstance(vv, (int, float)) and kk.startswith(("test_", "val_"))}
              for k, v in m.items()}).T

In [ ]:
!{SURGE} bench -b plasma.qlknn_transport -m sklearn.random_forest,pytorch.residual_mlp,kumo.tabular_regressor --seeds 1 2>&1 | grep -E "^\[(PASS|FAIL)\]|Leaderboard|test_r2" | head -n 12
!{GALLERY} --only leaderboard --benchmark plasma.qlknn_transport --reports benchmark_reports --modes light --out gallery 2>&1 | tail -n 2
p = next(Path("gallery").glob("leaderboard*light*.png"), None) or next(Path("gallery").glob("leaderboard*.png"), None)
display(Image(filename=str(p), width=900)) if p else print("no leaderboard figure")

## 4 · Report & deploy

`surge report` builds a single self-contained HTML page from the receipts: status tiers, gates, citations,
dataset previews and the sortable tables. Deployment is the saved model: reload it and predict on new rows.

In [ ]:
!{SURGE} report --out leaderboard.html 2>&1 | tail -n 2
HTML(Path("leaderboard.html").read_text()[:2_000_000])

In [ ]:
from surge.io.load_compat import load_model_compat
entry = m["qlknn_residual_mlp"]
model = load_model_compat(Path("runs/qlknn_colab/models/qlknn_residual_mlp.joblib"), {"backend": "pytorch"})
X_new = df[cols].sample(5, random_state=0)
print("new rows -> predicted efeITG:", np.round(np.asarray(model.predict(X_new.to_numpy())).ravel(), 3))

## 5 · A second fusion dataset in two minutes — SOLPS-NN divertor exhaust

The SOLPS-NN record (Dasbach & Wiesen 2023, Zenodo 10.5281/zenodo.19237127, CC BY 4.0) is a 17.6 GB
archive of SOLPS-ITER simulations of a size-scaled JET divertor plasma. The SURGE loader reads the archive's
table of contents over HTTP byte ranges and pulls only the members a task needs; the exhaust-scalar task
needs under 2 MB. The benchmark maps eight machine and control parameters to the power into the SOL, the
peak heat flux and the particle flux at the outer target, with the record's own Sobol-train / random-test
split. The published surrogates (tuned five-fold networks) reach a 13 % median relative error on the peak
heat flux; Kumo Tabular, with the 5,756 training runs as context and no training, does better.

In [ ]:
!{SURGE} bench -b plasma.solpsnn_exhaust_scalars -m kumo.tabular_regressor --no-save 2>&1 | grep -E "^\[(PASS|FAIL)\]|median_rel_err" | head -n 3

In [ ]:
from surge.benchmarks.loaders import solpsnn as ds
from surge.model import MODEL_REGISTRY
from surge.viz.solpsnn import scalar_parity_figure

tr, te = ds.load_solpsnn("train", with_te=False), ds.load_solpsnn("test", with_te=False)
preds = {}
for name in ("psol", "pwmxap", "fnixap"):
    k = MODEL_REGISTRY.create("kumo.tabular_regressor", target_transform="log10", num_estimators=4)
    k.fit(tr.X_log, tr.scalars[name])
    preds[name] = (te.scalars[name], np.asarray(k.predict(te.X_log)).ravel())
fig = scalar_parity_figure(preds, labels={"psol": "P_SOL", "pwmxap": "q_peak,ot", "fnixap": "Gamma_D+,ot"},
                           units={n: ds.SCALAR_TARGETS[n][1] for n in preds})
display(fig)

## Where to go next

- The [README](https://github.com/asvillar/SURGE#readme) walks through QLKNN10D (290M QuaLiKiz fluxes),
  ConStellaration stellarator equilibria and the SOLPS-NN scenario atlas; the
  [benchmark ledger](https://github.com/asvillar/SURGE/blob/release/0.1.0/docs/benchmarks/README.md) lists
  every benchmark with its status, gate and citation.
- `surge init --data your.csv --target y --yes --out spec.yaml` is the same wizard on your own data;
  `docs/BUILD_YOUR_OWN_SURROGATE.md` is the playbook.
- Please cite SURGE via [DOE CODE 179819](https://www.osti.gov/doecode/biblio/179819), DOI
  10.11578/dc.20260422.5.